In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import altair as alt
import pyarrow

In [2]:
alt.data_transformers.enable("vegafusion")

DataTransformerRegistry.enable('vegafusion')

In [3]:
df_hora = pd.read_csv('data/dados_hora.csv')
df_diarios = pd.read_csv('data/dados_diarios.csv')

In [4]:
df_hora['Resto (kWh)'] = df_hora['Rede Distribuição (kWh)'] - (df_hora['Eólica (kWh)'] + df_hora['Fotovoltaica (kWh)'] + df_hora['Hídrica (kWh)'] )

In [5]:
df_hora.describe()

,Cogeração (kWh),Eólica (kWh),Fotovoltaica (kWh),Hídrica (kWh),Outras Tecnologias (kWh),Rede Distribuição (kWh),Baixa Tensão (kWh),Média Tensão (kWh),Alta Tensão (kWh),Muito Alta Tensão (kWh),Dia,Mês,Ano,Mercado (kWh),Regime Especial (kWh),Total (kWh) (Consumido),Total (kWh) (Produzido),Resto (kWh)
count,109981.000000,1.099810e+05,109981.000000,109981.000000,109981.000000,1.099810e+05,1.099810e+05,109981.000000,109981.000000,109981.000000,109981.000000,109981.000000,109981.000000,1.099810e+05,1.099810e+05,1.099810e+05,1.099810e+05,109981.000000
mean,42107.022813,3.839374e+05,9399.109369,23595.893400,65396.774071,5.244362e+05,7.622991e+05,452311.478722,197598.470101,71292.390699,15.661051,6.298706,2024.084687,9.604390e+05,5.244274e+05,1.483501e+06,1.484866e+06,107503.796884
std,25470.863331,2.830175e+05,13271.313071,17321.372333,23767.494822,2.832140e+05,2.141918e+05,120512.717902,19256.419943,17242.062001,8.804726,3.533235,0.895114,3.616615e+05,2.832203e+05,2.786397e+05,2.783659e+05,26403.338227
min,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,1.000000,1.000000,2023.000000,-2.267620e+05,0.000000e+00,0.000000e+00,0.000000e+00,0.000000
25%,19103.750000,1.496928e+05,0.250000,6168.500000,43854.468000,2.944831e+05,6.084690e+05,350009.521600,187234.566600,59152.361900,8.000000,3.000000,2023.000000,7.246130e+05,2.944607e+05,1.252330e+06,1.253547e+06,90500.750000
50%,47480.750000,3.193232e+05,183.250000,21362.000000,62524.546000,4.612802e+05,7.273177e+05,432361.067400,199737.935800,72896.314800,16.000000,6.000000,2024.000000,9.711140e+05,4.612802e+05,1.473203e+06,1.474199e+06,106883.789000
75%,63679.250000,5.651491e+05,18112.750000,40631.250000,83214.500000,7.011767e+05,8.786290e+05,554676.295600,210840.052400,85661.813000,23.000000,9.000000,2025.000000,1.214783e+06,7.011767e+05,1.668029e+06,1.669938e+06,123777.750000
max,102841.000000,1.238478e+06,50510.500000,95408.000000,144804.805006,1.466503e+06,1.729600e+06,761619.116200,244902.224000,109840.462400,31.000000,12.000000,2026.000000,2.178261e+06,1.466503e+06,2.561410e+06,2.561410e+06,201650.053023


In [6]:
tecnologias = ['Eólica (kWh)', 'Fotovoltaica (kWh)', 'Hídrica (kWh)', 'Resto (kWh)']

# 2. "Derreter" o DataFrame
df_long = df_hora.melt(
    id_vars=['Data/Hora'], 
    value_vars=tecnologias,
    var_name='Tecnologia', 
    value_name='kWh'
)

# 3. Criar o gráfico
chart = alt.Chart(df_long).mark_area().encode(
    alt.X('yearmonth(Data/Hora):T').axis(format='%b %Y', title='Mês/Ano'),
    alt.Y('sum(kWh):Q').title('Produção Total (kWh)'),
    
    # Adicionamos o 'sort' aqui para ordenar pela soma de kWh
    alt.Color('Tecnologia:N').scale(scheme='inferno').sort(
        alt.EncodingSortField(field='kWh', op='sum', order='descending')
    ),
    
    alt.Order('sum(kWh):Q', sort='descending'),
    
    tooltip=['yearmonth(Data/Hora)', 'Tecnologia', 'sum(kWh)']
).properties(
    width=800,
    height=400,
    title='Evolução Mensal da Produção de Energia'
).interactive()

chart.show()

alt.Chart(...)

In [7]:
dfII = df_hora.copy()
dfII = dfII.groupby(by=[dfII['Ano'],dfII['Mês']]).sum().reset_index()
dfII = dfII[['Ano', 'Mês', 'Eólica (kWh)', 'Fotovoltaica (kWh)', 'Hídrica (kWh)', 'Resto (kWh)']]
dfII['Ano'] = dfII['Ano'].astype(str)
dfII['Mes/Ano'] = dfII['Mês'].astype(str) + '/' + dfII['Ano'].astype(str)
anos_legend= list(dfII.apply(
    lambda x: str(x['Ano']) if x['Mês'] == 6 else "", axis=1
))
r = "Eólica (kWh)"
anos_legend[-1] = "2026"
fig = px.bar_polar(
    dfII,
    r= r,
    theta="Mes/Ano",
    color="Ano",
    labels="Ano",
    title=f"Evolução Mensal da Produção de Energia: {r}",
    color_discrete_sequence=px.colors.qualitative.D3
).update_layout(
    showlegend=True,
    coloraxis_showscale=False,
    legend=dict(
        title="Ano de Produção",
        font=dict(size=12),
        # Isto garante que a legenda não fica preta se o fundo for escuro
        itemsizing='constant' 
    ),
    polar=dict(hole = 0.2,
          angularaxis=dict(
                type="category",
                # IMPORTANTE: O array de categorias tem de ser a coluna theta completa
                categoryarray=dfII['Mes/Ano'].tolist(),
                categoryorder="array",
                # O período tem de ser o número total de fatias para fechar o círculo
                period=len(dfII),
                tickvals=dfII['Mes/Ano'].tolist(),
                # O texto é que leva a lista com vazios
                ticktext=anos_legend,
                direction="clockwise",
                rotation=90,
                tickfont=dict(size=18, family='Arial', style='italic')
        ),  
        radialaxis=dict(
            showticklabels=True, 
            range=[0, dfII[r].max()*1.02],
            nticks=5, 
            tickfont=dict(size=18, family='Arial'),
            linecolor='black', 
            linewidth=1,
            layer='above traces', # Mudei para 'below' para as barras não taparem os números
            gridcolor='lightgrey'
        ),   
    ),
    
    height=600,
    width=800,
    margin=dict(b=30, t=80, l=0, r=0),
    )
fig.show()


    

In [8]:
tecnologias = ['Eólica (kWh)', 'Fotovoltaica (kWh)', 'Hídrica (kWh)', 'Resto (kWh)']

# O melt mantém 'Ano' e 'Mês' e transforma as colunas de tecnologia em linhas
df_new = df_hora.melt(
    id_vars=['Ano', 'Mês'], 
    value_vars=tecnologias,
    var_name='Tecnologia', 
    value_name='Producao'
)

dfIII = df_new.groupby(by=[df_new['Ano'],df_new['Mês'],df_new['Tecnologia']]).sum().reset_index()

dfIII['Ano'] = dfIII['Ano'].astype(str)
dfIII['Mes/Ano'] = dfIII['Mês'].astype(str) + '/' + dfIII['Ano'].astype(str)
r = "Producao"
meses = dict(zip(range(1,13), ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']))
anos_legend= list(dfIII.apply(
    lambda x: meses[x['Mês']]+ "/" + x['Ano'] if x['Mês'] %2 != 0 else "", axis=1
))
anos_legend[-1] = "2026"
fig = px.bar_polar(
    dfIII,
    r= r,
    theta="Mes/Ano",
    color="Tecnologia",
    labels="Ano",
    title=f"Evolução Mensal da Produção de Energia: {r}",
    color_discrete_sequence=px.colors.qualitative.Set3).update_layout(
    showlegend=True,
    coloraxis_showscale=False,
    legend=dict(
        title="Ano de Produção",
        font=dict(size=12),
        # Isto garante que a legenda não fica preta se o fundo for escuro
        itemsizing='constant' 
    ),
    polar=dict(hole = 0.2,
          angularaxis=dict(
                type="category",
                # IMPORTANTE: O array de categorias tem de ser a coluna theta completa
                categoryarray=dfIII['Mes/Ano'].tolist(),
                categoryorder="array",
                # O período tem de ser o número total de fatias para fechar o círculo
                period=len(dfIII['Mes/Ano'].unique()),
                tickvals=dfIII['Mes/Ano'].tolist(),
                # O texto é que leva a lista com vazios
                ticktext=anos_legend,
                direction="clockwise",
                rotation=90,
                tickfont=dict(size=18, family='Arial', style='italic')
        ),  
        radialaxis=dict(
            showticklabels=True, 
            range=[0, dfIII[r].max()*1.02],
            nticks=5, 
            tickfont=dict(size=18, family='Arial'),
            linecolor='black', 
            linewidth=1,
            layer='above traces', # Mudei para 'below' para as barras não taparem os números
            gridcolor='lightgrey'
        ),   
    ),
    
    height=1000,
    width=1000,
    margin=dict(b=30, t=80, l=100, r=60),
    )

fig.show()

In [21]:
dfIII.to_csv('data/dados_producao_mesano.csv', index=False)

In [ ]:
# 1. Criar dados fictícios (3 anos)
years = [2023, 2024, 2025]
months = np.arange(1, 13)
data = []

for y in years:
    for m in months:
        # Valor base + sazonalidade + ruído
        val = 10 + np.sin(m/2) * 5 + np.random.rand() * 2
        data.append({'year': y, 'month': m, 'value': val})

df = pd.DataFrame(data)

# 2. Normalização e Offset (o segredo da espiral)
# No R, o ymin era o 'y'. Aqui criamos a base da barra que sobe com o tempo.
max_val = df['value'].max()
df = df.sort_values(['year', 'month']).reset_index(drop=True)

# r_base faz com que cada mês suba um pouco, criando a espiral
df['r_base'] = np.arange(len(df)) * 0.8 
df['r_top'] = df['r_base'] + (df['value'] / max_val * 5) # Altura da barra

In [34]:
fig = go.Figure()
paleta = px.colors.sequential.Viridis
# Adicionamos uma série por ano para manter as cores


# O enumerate devolve o índice (i) e o valor (year)
for i, year in enumerate(df['year'].unique()):
    temp = df[df['year'] == year]
    
    fig.add_trace(go.Barpolar(
        r=temp['value'],
        theta=temp['month'] * 30,
        base=temp['r_base'],
        name=str(year),
        # Usamos o i para escolher a cor: 
        # i=0 (2023) -> paleta[0]
        # i=1 (2024) -> paleta[2] (saltamos para variar mais a cor)
        marker_color=paleta[i * 2 % len(paleta)], 
        marker_line_color="black",
        thetaunit="degrees"
    ))

fig.update_layout(
    polar=dict( hole =0.2,
        angularaxis=dict(
            # Onde os nomes aparecem (em graus)
            tickvals=[30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330, 360],
            ticktext=['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 
                      'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez'],
            direction="clockwise",
            rotation=90 # Janeiro no topo
        )
    )
)

fig.show()

KeyError: 'year'

In [36]:
"""
ano = dfIII['Ano']
mes = dfIII['Mês']
data = []

for y in ano:
    for m in mes:
        # Valor base + sazonalidade + ruído
        #val = 10 + np.sin(m/2) * 5 + np.random.rand() * 2
        data.append({'year': y, 'month': m, 'value': val})

df = pd.DataFrame(data)
"""
# 2. Normalização e Offset (o segredo da espiral)
# No R, o ymin era o 'y'. Aqui criamos a base da barra que sobe com o tempo.
dfIV = dfIII.groupby(by=[dfIII['Ano'], dfIII['Mês']]).sum().reset_index()
dfIV = dfIV[['Ano', 'Mês', 'Producao']]
max_val = dfIV['Producao'].max()
dfIV = dfIV.sort_values(['Ano', 'Mês']).reset_index(drop=True)

# r_base faz com que cada mês suba um pouco, criando a espiral
dfIV['r_base'] = np.arange(len(dfIV)) * 20
dfIV['r_top'] = dfIV['r_base'] + (dfIV['Producao'] / max_val * 5)

In [37]:
fig = go.Figure()
paleta = px.colors.sequential.Viridis
# Adicionamos uma série por ano para manter as cores


# O enumerate devolve o índice (i) e o valor (year)
for i, year in enumerate(dfIV['Ano'].unique()):
    temp = dfIV[dfIV['Ano'] == year]
    
    fig.add_trace(go.Barpolar(
        r=temp['Producao'],
        theta=temp['Mês'] * 30,
        base=temp['r_base'],
        name=str(year),
        # Usamos o i para escolher a cor: 
        # i=0 (2023) -> paleta[0]
        # i=1 (2024) -> paleta[2] (saltamos para variar mais a cor)
        marker_color=paleta[i * 2 % len(paleta)], 
        marker_line_color="black",
        thetaunit="degrees"
    ))

fig.update_layout(
    polar=dict( hole = 0.5,
        angularaxis=dict(
            # Onde os nomes aparecem (em graus)
            tickvals=[30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330, 360],
            ticktext=['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 
                      'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez'],
            direction="clockwise",
            rotation=90 # Janeiro no topo
        )
    )
)

fig.show()

In [68]:
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
import numpy as np

# --- 1. Preparação dos Dados para a Espiral ---
# Ordenamos os dados cronologicamente (essencial para a espiral)

df_spiral = dfIII.groupby(by=[dfIII['Ano'], dfIII['Mês']]).sum().reset_index()
df_spiral = df_spiral[['Ano', 'Mês', 'Producao']]
max_val = df_spiral['Producao'].max()
df_spiral = df_spiral.sort_values(['Ano', 'Mês']).reset_index(drop=True)
df_spiral = df_spiral[:-1]
df_spiral = df_spiral[:-1]

# r_base faz com que cada mês suba um pouco, criando a espiral
df_spiral['r_base'] = np.arange(len(df_spiral)) * 20
df_spiral['r_top'] = df_spiral['r_base'] + (df_spiral['Producao'] / max_val * 5)

# Normalizamos a produção para que o tamanho das barras seja proporcional
# mas caiba bem na espiral.
norm_factor = df_spiral['Producao'].max() * 0.1 

# Criamos a base da barra que aumenta continuamente com o tempo (a espiral)
# Isto faz com que Dezembro e Janeiro do ano seguinte não se sobreponham.
df_spiral['r_base'] = np.arange(len(df_spiral)) 
df_spiral['r_top'] = df_spiral['r_base'] + (df_spiral['Producao'] / norm_factor)

r_total_max = df_spiral['r_top'].max()

# --- 2. Criação do Gráfico ---
fig = go.Figure()

# Usamos loops para adicionar cores e rótulos de forma organizada
# (Simulando o interaction(month, year) do R)
for year in df_spiral['Ano'].unique():
    temp = df_spiral[df_spiral['Ano'] == year]
    
    fig.add_trace(go.Barpolar(
        r=temp['Producao'] / norm_factor, # Comprimento da barra
        # Convertemos meses (1-12) em graus (0-360) para ocupar o círculo todo
        theta=temp['Mês'] * (360/12), 
        base=temp['r_base']*2,          # O SEGREDO: O offset que cria a espiral
        name=str(year),
        customdata=temp['Producao'],
        hovertemplate="<b>Produção:</b> %{customdata:.2e} kWh<extra></extra>",
        thetaunit="degrees"
    ))

# --- 3. Layout Estilizado (Limpo e Focado na Forma) ---
r_real_max = (df_spiral['r_base'].max() * 2) + (df_spiral['Producao'].max() / norm_factor)
fig.update_layout(
    title="Spiral Histogram: Evolução da Produção Eólica (kWh)",
    font_size=16,
    polar=dict(
        hole=0.2,
        angularaxis=dict(
            # Configuração para que os nomes dos meses apareçam nos graus certos
            tickvals=[30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330, 360],
            ticktext=['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez'],
            direction="clockwise",
            rotation=90, # Janeiro no topo
            tickfont=dict(size=20, family='Arial', style='italic')
        ),
        # Removemos os círculos e números do eixo radial para ficar limpo
        
        radialaxis=dict(
            showticklabels=False,
            ticks="",
            showline=False,
            range=[0, r_real_max * 1.1] # Dá espaço para as últimas barras
        )
    ),
    # Mostramos a legenda apenas para os anos
    showlegend=True,
    legend=dict(title="Ano de Produção", font=dict(size=12), itemsizing='constant'),
    height=900,
    width=1100,
    margin=dict(b=30, t=80, l=0, r=10),
)

# Se quiseres a barra de cores (gradiente) como no exemplo 1, 
# terias de usar uma abordagem ligeiramente diferente com go.Scatterpolar
# mas esta com Barpolar é mais interativa.

fig.show()

In [69]:
dfIII[dfIII['Tecnologia'] == 'Eólica (kWh)']

,Ano,Mês,Tecnologia,Producao,Mes/Ano
0,2023,1,Eólica (kWh),1.406324e+09,1/2023
4,2023,2,Eólica (kWh),1.166547e+09,2/2023
8,2023,3,Eólica (kWh),1.150473e+09,3/2023
12,2023,4,Eólica (kWh),8.407546e+08,4/2023
16,2023,5,Eólica (kWh),1.115504e+09,5/2023
20,2023,6,Eólica (kWh),6.450615e+08,6/2023
24,2023,7,Eólica (kWh),9.016805e+08,7/2023
28,2023,8,Eólica (kWh),9.742139e+08,8/2023
32,2023,9,Eólica (kWh),7.148228e+08,9/2023
36,2023,10,Eólica (kWh),1.332436e+09,10/2023


In [87]:
def create_spiral_histogram(tec:str):
    if tec not in dfIII['Tecnologia'].unique():
        raise ValueError(f"Tecnologia '{tec}' não encontrada. Opções: {dfIII['Tecnologia'].unique()}")
    df_spiral = dfIII[dfIII['Tecnologia'] == tec]
    df_spiral = df_spiral[['Ano', 'Mês', 'Producao']]
    max_val = df_spiral['Producao'].max()
    df_spiral = df_spiral.sort_values(['Ano', 'Mês']).reset_index(drop=True)
    df_spiral = df_spiral[:-1]
    df_spiral = df_spiral[:-1]

    # r_base faz com que cada mês suba um pouco, criando a espiral
    df_spiral['r_base'] = np.arange(len(df_spiral)) * 20
    df_spiral['r_top'] = df_spiral['r_base'] + (df_spiral['Producao'] / max_val * 5)

    # Normalizamos a produção para que o tamanho das barras seja proporcional
    # mas caiba bem na espiral.
    norm_factor = df_spiral['Producao'].max() * 0.05

    # Criamos a base da barra que aumenta continuamente com o tempo (a espiral)
    # Isto faz com que Dezembro e Janeiro do ano seguinte não se sobreponham.
    df_spiral['r_base'] = np.arange(len(df_spiral)) 
    df_spiral['r_top'] = df_spiral['r_base'] + (df_spiral['Producao'] / norm_factor)

    r_total_max = df_spiral['r_top'].max()

    # --- 2. Criação do Gráfico ---
    fig = go.Figure()

    # Usamos loops para adicionar cores e rótulos de forma organizada
    # (Simulando o interaction(month, year) do R)
    for year in df_spiral['Ano'].unique():
        temp = df_spiral[df_spiral['Ano'] == year]
        
        fig.add_trace(go.Barpolar(
            r=temp['Producao'] / norm_factor, # Comprimento da barra
            # Convertemos meses (1-12) em graus (0-360) para ocupar o círculo todo
            theta=temp['Mês'] * (360/12), 
            base=temp['r_base']*2,          # O SEGREDO: O offset que cria a espiral
            name=str(year),
            customdata=temp['Producao'],
            hovertemplate="<b>Produção:</b> %{customdata:.2e} kWh<extra></extra>",
            thetaunit="degrees"
        ))

    # --- 3. Layout Estilizado (Limpo e Focado na Forma) ---
    r_real_max = (df_spiral['r_base'].max() * 2) + (df_spiral['Producao'].max() / norm_factor)
    fig.update_layout(
        title="Spiral Histogram: Evolução da Produção Eólica (kWh)",
        font_size=16,
        polar=dict(
            hole=0.2,
            angularaxis=dict(
                # Configuração para que os nomes dos meses apareçam nos graus certos
                tickvals=[30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330, 360],
                ticktext=['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez'],
                direction="clockwise",
                rotation=120, # Janeiro no topo
                tickfont=dict(size=20, family='Arial', style='italic')
            ),
            # Removemos os círculos e números do eixo radial para ficar limpo
            
            radialaxis=dict(
                showticklabels=False,
                ticks="",
                showline=False,
                range=[0, r_real_max * 1.05] # Dá espaço para as últimas barras
            )
        ),
        # Mostramos a legenda apenas para os anos
        showlegend=True,
        legend=dict(title="Ano de Produção", font=dict(size=12), itemsizing='constant'),
        height=900,
        width=1100,
        margin=dict(b=30, t=80, l=0, r=10),
    )

    # Se quiseres a barra de cores (gradiente) como no exemplo 1, 
    # terias de usar uma abordagem ligeiramente diferente com go.Scatterpolar
    # mas esta com Barpolar é mais interativa.

    fig.show()

In [88]:
create_spiral_histogram('Eólica (kWh)') 